# StatPitch
## Notebook 7 — Live Matches with API-Football

---

This notebook adds two things:

1. **Live match fetching** — query API-Football for today's scheduled international fixtures and generate predictions for each one automatically.
2. **A new `/today` endpoint** in the StatPitch API — so your frontend can show today's matches with predictions with a single call.

**Setup needed (one time):**
1. Go to [rapidapi.com](https://rapidapi.com) and create a free account
2. Search for **API-Football** and subscribe to the free plan (100 requests/day)
3. Copy your RapidAPI key from the dashboard
4. Paste it in the cell below

**For deployment:** Add `RAPIDAPI_KEY` as an environment variable in your Render service dashboard — never hardcode it in your code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, sys
import requests
import pandas as pd
from datetime import datetime

SAVE_DIR    = '/content/drive/MyDrive/StatPitch'
PROJECT_DIR = os.path.join(SAVE_DIR, 'statpitch_api')
os.chdir(SAVE_DIR)

# ─── Paste your RapidAPI key here ────────────────────────────────────────────
RAPIDAPI_KEY = 'YOUR_KEY_HERE'
# ─────────────────────────────────────────────────────────────────────────────

RAPIDAPI_HOST = 'api-football-v1.p.rapidapi.com'
BASE_URL      = f'https://{RAPIDAPI_HOST}/v3'
HEADERS       = {'X-RapidAPI-Key': RAPIDAPI_KEY, 'X-RapidAPI-Host': RAPIDAPI_HOST}

print('Setup complete.')
print(f'Working in: {os.getcwd()}')
print(f'Project dir: {PROJECT_DIR}')

---
## Step 1 — Test API connection

In [ ]:
def api_get(endpoint, params=None):
    resp = requests.get(f'{BASE_URL}/{endpoint}', headers=HEADERS, params=params, timeout=10)
    data = resp.json()
    if 'errors' in data and data['errors']:
        raise ValueError(f'API error: {data["errors"]}')
    return data.get('response', [])

# Check remaining requests
resp = requests.get(f'{BASE_URL}/status', headers=HEADERS)
status = resp.json()
account = status.get('response', {}).get('account', {})
sub     = status.get('response', {}).get('subscription', {})
reqs    = status.get('response', {}).get('requests', {})

print('API-Football connection successful!')
print(f'  Plan:               {sub.get("plan", "unknown")}')
print(f'  Requests today:     {reqs.get("current", "?")}')
print(f'  Requests remaining: {reqs.get("limit_day", 100) - reqs.get("current", 0)}')

---
## Step 2 — Fetch today's international fixtures

In [ ]:
# International competition league IDs in API-Football
INTL_LEAGUES = {
    1:   'FIFA World Cup',
    4:   'Euro Championship',
    5:   'UEFA Nations League',
    6:   'Copa America',
    7:   'African Cup of Nations',
    8:   'AFC Asian Cup',
    9:   'Copa America',
    10:  'CONCACAF Gold Cup',
    848: 'World Cup Qualification Europe',
    849: 'World Cup Qualification South America',
    860: 'World Cup Qualification Africa',
    861: 'World Cup Qualification Asia',
    882: 'World Cup Qualification CONCACAF',
    1002:'World Cup Qualification Oceania',
    531: 'UEFA Nations League',
}

# Team name mapping: API-Football name → StatPitch name
NAME_MAP_API = {
    'Korea Republic':      'South Korea',
    'Korea DPR':           'North Korea',
    'IR Iran':             'Iran',
    'Czechia':             'Czech Republic',
    'Bosnia-Herzegovina':  'Bosnia-Herzegovina',
    'United States':       'United States',
    'Trinidad And Tobago': 'Trinidad and Tobago',
    "Cote d'Ivoire":       'Ivory Coast',
    'Cape Verde Islands':  'Cape Verde',
    'Congo DR':            'DR Congo',
    'Kyrgyz Republic':     'Kyrgyzstan',
    'Eswatini':            'Swaziland',
}

def normalize_team(name):
    return NAME_MAP_API.get(name, name)

def get_fixtures_for_date(date_str):
    all_fixtures = []
    for league_id, league_name in INTL_LEAGUES.items():
        try:
            fixtures = api_get('fixtures', {'date': date_str, 'league': league_id})
            for fix in fixtures:
                all_fixtures.append({
                    'fixture_id':   fix['fixture']['id'],
                    'date':         fix['fixture']['date'][:10],
                    'time':         fix['fixture']['date'][11:16],
                    'home_team':    normalize_team(fix['teams']['home']['name']),
                    'away_team':    normalize_team(fix['teams']['away']['name']),
                    'home_logo':    fix['teams']['home']['logo'],
                    'away_logo':    fix['teams']['away']['logo'],
                    'league':       league_name,
                    'league_id':    league_id,
                    'status':       fix['fixture']['status']['short'],
                    'home_score':   fix['goals']['home'],
                    'away_score':   fix['goals']['away'],
                    'venue':        fix['fixture']['venue']['name'],
                    'country':      fix['league']['country'],
                })
            if fixtures:
                time.sleep(0.3)
        except Exception:
            pass
    return all_fixtures

TODAY = datetime.now().strftime('%Y-%m-%d')
print(f'Fetching international fixtures for {TODAY}...')
fixtures = get_fixtures_for_date(TODAY)

print(f'Found {len(fixtures)} international matches today')
if fixtures:
    df_fix = pd.DataFrame(fixtures)
    print(df_fix[['time','home_team','away_team','league','status']].to_string(index=False))
else:
    print('No international fixtures today — try a specific date:')
    print('  fixtures = get_fixtures_for_date("2026-06-15")  # example WC 2026 date')

---
## Step 3 — Generate StatPitch predictions for today's matches

In [ ]:
# Load predictor
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import importlib
import predictor
importlib.reload(predictor)

available_teams = set(predictor.get_teams())
print(f'StatPitch has stats for {len(available_teams)} teams')
os.chdir(SAVE_DIR)

def predict_fixture(fix):
    home = fix['home_team']
    away = fix['away_team']

    if home not in available_teams or away not in available_teams:
        return {**fix, 'prediction': None,
                'note': f'Not in database: {home if home not in available_teams else away}'}

    pred = predictor.predict(home, away, is_neutral=True,
                              match_date=fix.get('date'))
    return {**fix, 'prediction': pred}

print('Generating predictions...')
results = [predict_fixture(f) for f in fixtures]
predicted = [r for r in results if r['prediction']]
skipped   = [r for r in results if not r['prediction']]

print(f'  Predicted: {len(predicted)}')
print(f'  Skipped:   {len(skipped)}')

In [ ]:
# Pretty print today's predictions
if not predicted:
    print('No predictions today — fetch a future date with international matches.')
else:
    print(f'StatPitch predictions for {TODAY}')
    print('=' * 65)
    for r in predicted:
        p  = r['prediction']
        mr = p['match_result']
        ou = p['over_under']
        bt = p['btts']
        eg = p['expected_goals']
        ht = r['home_team'][:20]
        at = r['away_team'][:20]
        print(f"\n  {r['time']}  {r['league']}")
        print(f'  {ht:<22} vs  {at}')
        print(f'  xG: {eg["home"]:.2f} — {eg["away"]:.2f}')
        print(f'  1X2:  H {mr["home_win"]*100:.0f}%   D {mr["draw"]*100:.0f}%   A {mr["away_win"]*100:.0f}%')
        print(f'  O2.5: {ou["over_2_5"]*100:.0f}%   BTTS: {bt["yes"]*100:.0f}%')
        top1 = p['correct_score'][0]
        print(f'  Most likely score: {top1["score"]} ({top1["probability"]*100:.1f}%)')

---
## Step 4 — Write `live_fetcher.py`

This module is imported by `main.py` to power the `/today` endpoint.

In [ ]:
%%writefile /content/drive/MyDrive/StatPitch/statpitch_api/live_fetcher.py
# live_fetcher.py  --  StatPitch  --  fetches today's international fixtures
import os, time, requests
from datetime import datetime
import predictor

RAPIDAPI_KEY  = os.getenv('RAPIDAPI_KEY', '')
RAPIDAPI_HOST = 'api-football-v1.p.rapidapi.com'
BASE_URL      = f'https://{RAPIDAPI_HOST}/v3'

INTL_LEAGUES = {
    1: 'FIFA World Cup', 4: 'Euro Championship',
    5: 'UEFA Nations League', 6: 'Copa America',
    7: 'African Cup of Nations', 8: 'AFC Asian Cup',
    10: 'CONCACAF Gold Cup', 848: 'WC Qual Europe',
    849: 'WC Qual South America', 860: 'WC Qual Africa',
    861: 'WC Qual Asia', 882: 'WC Qual CONCACAF',
}

NAME_MAP = {
    'Korea Republic': 'South Korea', 'Korea DPR': 'North Korea',
    'IR Iran': 'Iran', 'Czechia': 'Czech Republic',
    'United States': 'United States', "Cote d'Ivoire": 'Ivory Coast',
    'Cape Verde Islands': 'Cape Verde', 'Congo DR': 'DR Congo',
    'Eswatini': 'Swaziland', 'Trinidad And Tobago': 'Trinidad and Tobago',
}

def _norm(name):
    return NAME_MAP.get(name, name)


def get_fixtures(date_str=None):
    if not RAPIDAPI_KEY:
        return []
    if date_str is None:
        date_str = datetime.now().strftime('%Y-%m-%d')

    headers  = {'X-RapidAPI-Key': RAPIDAPI_KEY, 'X-RapidAPI-Host': RAPIDAPI_HOST}
    fixtures = []

    for league_id, league_name in INTL_LEAGUES.items():
        try:
            resp = requests.get(
                f'{BASE_URL}/fixtures',
                headers=headers,
                params={'date': date_str, 'league': league_id},
                timeout=8,
            )
            for fix in resp.json().get('response', []):
                fixtures.append({
                    'fixture_id': fix['fixture']['id'],
                    'date':       fix['fixture']['date'][:10],
                    'time':       fix['fixture']['date'][11:16],
                    'home_team':  _norm(fix['teams']['home']['name']),
                    'away_team':  _norm(fix['teams']['away']['name']),
                    'league':     league_name,
                    'status':     fix['fixture']['status']['short'],
                    'home_score': fix['goals']['home'],
                    'away_score': fix['goals']['away'],
                    'venue':      fix['fixture']['venue']['name'],
                })
            time.sleep(0.3)
        except Exception:
            pass
    return fixtures


def get_today_predictions(date_str=None):
    if date_str is None:
        date_str = datetime.now().strftime('%Y-%m-%d')

    fixtures = get_fixtures(date_str)
    available = set(predictor.get_teams())
    results   = []

    for fix in fixtures:
        home = fix['home_team']
        away = fix['away_team']
        if home in available and away in available:
            pred = predictor.predict(home, away, is_neutral=True, match_date=fix['date'])
            results.append({**fix, 'prediction': pred, 'predicted': True})
        else:
            missing = home if home not in available else away
            results.append({**fix, 'prediction': None, 'predicted': False,
                             'note': f'Not in database: {missing}'})

    return {
        'date':               date_str,
        'fetched':            bool(RAPIDAPI_KEY),
        'total_fixtures':     len(fixtures),
        'predicted_fixtures': sum(r['predicted'] for r in results),
        'matches':            results,
    }

---
## Step 5 — Update `main.py` with StatPitch branding and `/today` endpoint

In [ ]:
%%writefile /content/drive/MyDrive/StatPitch/statpitch_api/main.py
# main.py  --  StatPitch API
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import predictor
import live_fetcher

app = FastAPI(
    title='StatPitch API',
    description='AI-powered football match predictions — powered by StatPitch',
    version='3.0.0',
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


class PredictRequest(BaseModel):
    home_team:   str
    away_team:   str
    is_neutral:  Optional[bool] = True
    match_date:  Optional[str]  = None   # format YYYY-MM-DD


def _validate(home: str, away: str):
    teams = predictor.get_teams()
    if home not in teams:
        raise HTTPException(status_code=404, detail=f'Team not found: {home}')
    if away not in teams:
        raise HTTPException(status_code=404, detail=f'Team not found: {away}')
    if home == away:
        raise HTTPException(status_code=400, detail='Teams must be different')


@app.get('/')
def root():
    return {
        'service':  'StatPitch API',
        'version':  '3.0.0',
        'endpoints': ['/health', '/teams', '/predict', '/today', '/today/{date}', '/docs'],
    }


@app.get('/health')
def health():
    return {'status': 'ok', 'teams': len(predictor.TEAM_STATS), 'version': '3.0.0'}


@app.get('/teams')
def get_teams():
    ranked = sorted(
        [{'team': k, 'elo': round(v.get('elo', 1500), 1)}
         for k, v in predictor.TEAM_STATS.items()],
        key=lambda x: x['elo'], reverse=True,
    )
    return {'count': len(ranked), 'teams': ranked}


@app.post('/predict')
def predict_post(req: PredictRequest):
    _validate(req.home_team, req.away_team)
    return predictor.predict(req.home_team, req.away_team, req.is_neutral, req.match_date)


@app.get('/predict/{home_team}/{away_team}')
def predict_get(home_team: str, away_team: str, is_neutral: bool = True, match_date: str = None):
    _validate(home_team, away_team)
    return predictor.predict(home_team, away_team, is_neutral, match_date)


@app.get('/today')
def today():
    return live_fetcher.get_today_predictions()


@app.get('/today/{date}')
def today_date(date: str):
    return live_fetcher.get_today_predictions(date_str=date)

---
## Step 6 — Deploy

### Set the API key on Render

Never put your API key in code. In your Render service:

1. Go to your StatPitch service on [render.com](https://render.com)
2. Click **Environment** in the left sidebar
3. Add a new variable:
   - Key: `RAPIDAPI_KEY`
   - Value: your key from RapidAPI
4. Click **Save Changes** — Render redeploys automatically

### New endpoints available after deploy

```
GET /today                      Predictions for all international matches today
GET /today/2026-06-20           Predictions for a specific date
GET /predict/Brazil/France      Same as before
GET /docs                       Interactive API docs
```

### In your frontend

```javascript
// Get today's matches with predictions
const res = await fetch('https://your-app.onrender.com/today')
const data = await res.json()
// data.matches is an array of fixtures, each with a .prediction object
```

### Free tier limits

The free RapidAPI plan gives 100 requests/day. Each call to `/today` uses ~12–15 requests (one per international league). So you can serve `/today` about 7–8 times per day for free. If you need more, the $10/month plan gives unlimited requests.

---
> **StatPitch is now complete.** You have trained models, a live API, FBref xG enrichment, and today's match predictions.